# Milestone / Bonus Item 5 — Semantisches Chunking

Lädt das gespeicherte Transcript aus M1 und bereitet alles vor, was wir für semantisches
Chunking brauchen: Segmente laden, OpenAI-Client, Hilfsfunktionen.


In [5]:
import sys, os, json
sys.path.append("../backend")

from dotenv import load_dotenv
from openai import OpenAI

load_dotenv(dotenv_path="../.env")
client = OpenAI()

# Transcript aus M1 laden (dieselbe Datei wie in 02_milestone2_indexing.ipynb)
with open("../data/transcripts/E7W4OQfJWdw.json", "r", encoding="utf-8") as f:
    transcript_data = json.load(f)

segments = transcript_data["segments"]
video_id = transcript_data["video_id"]

print(f"Video-ID: {video_id}")
print(f"Anzahl roher Segmente: {len(segments)}")

Video-ID: E7W4OQfJWdw
Anzahl roher Segmente: 2502


## Baseline: Retrieval-Qualität mit aktuellem 30s-Zeitfenster-Chunking

Testet die 6 Fragen gegen search_video() (AKTUELLES Chunking), bevor wir irgendetwas ändern.
Ergebnis dient als Vergleichsbasis für später.


In [1]:
import sys
sys.path.append("../backend")
from tools import search_video

baseline_questions = [
    "What foods contain choline?",
    "How much creatine per day is recommended?",
    "What are anthocyanins found in?",
    "Does fasting affect brain function?",
    "What is the professor's academic title?",
    "How does gut health relate to the brain?",
]

baseline_results = {}
for q in baseline_questions:
    results = search_video(q, n_results=2)
    baseline_results[q] = results
    print(f"--- {q} ---")
    for r in results:
        print(f"[{r['start']:.1f}s-{r['end']:.1f}s]: {r['text'][:150]}...")
    print()

--- What foods contain choline? ---
[1622.8s-1654.6s]: However, food sources seem to be the best source of choline. And as with the EPAs and the omega-3s, there are plenty of foods that are non-animal-base...
[1870.0s-1900.1s]: the various foods that contain choline, and I try and get those foods on a semi-regular basis. I do supplement with something called alpha-GPC, which ...

--- How much creatine per day is recommended? ---
[2028.6s-2061.2s]: The first author is Roschel, R-O-S-C-H-E-L. We will provide a link to this study, rather, this review, excuse me, in the caption. This was published j...
[2124.1s-2154.5s]: in people that aren't getting creatine from animal sources. And there's some evidence detailed within the review that I just described, that creatine ...

--- What are anthocyanins found in? ---
[2219.0s-2250.0s]: The interesting thing about blueberries and other berries, blackberries, dark currants, any of these thin-skinned berries that are purpleish in color,...
[2250.0

## Baseline-Bewertung mit LLM-Judge (retrieval_relevance)

Lässt ein zweites LLM bewerten, ob die gefundenen Chunks die Frage tatsächlich beantworten.


In [2]:
from langchain_openai import ChatOpenAI

judge_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

def judge_relevance(question: str, chunks: list) -> str:
    if not chunks:
        return "NOT_RELEVANT: Kein Chunk gefunden"
    context = "\n---\n".join(c["text"] for c in chunks)
    prompt = f"""Frage: {question}

Gefundene Textausschnitte:
{context}

Beantworten diese Ausschnitte die Frage inhaltlich? Antworte NUR mit "RELEVANT" oder "NOT_RELEVANT", 
gefolgt von einem Doppelpunkt und einer kurzen Begründung."""
    return judge_llm.invoke(prompt).content

baseline_scores = {}
for q, chunks in baseline_results.items():
    verdict = judge_relevance(q, chunks)
    baseline_scores[q] = verdict
    print(f"--- {q} ---")
    print(verdict)
    print()

--- What foods contain choline? ---
RELEVANT: Die Textausschnitte nennen verschiedene Lebensmittel, die Cholin enthalten, wie Eier, Kartoffeln, Nüsse, Samen, Getreide und Obst.

--- How much creatine per day is recommended? ---
RELEVANT: Die Textausschnitte geben an, dass eine tägliche Einnahme von mindestens fünf Gramm Kreatin empfohlen wird, insbesondere für Personen, die keine tierischen Produkte konsumieren.

--- What are anthocyanins found in? ---
RELEVANT: The excerpts provide information about anthocyanins being found in blueberries, blackberries, and dark currants, and discuss their health benefits.

--- Does fasting affect brain function? ---
NOT_RELEVANT: Die Textausschnitte sprechen über Ernährung und deren Einfluss auf die Gehirnfunktion, jedoch wird das Thema Fasten nicht behandelt.

--- What is the professor's academic title? ---
NOT_RELEVANT: Kein Chunk gefunden

--- How does gut health relate to the brain? ---
RELEVANT: Die Textausschnitte erklären, wie ein gesunder Mik

## Semantisches Chunking: Segmente nach thematischer Ähnlichkeit gruppieren

Ersetzt die starre 30s-Zeitfenster-Logik durch Gruppierung nach Embedding-Ähnlichkeit
zwischen aufeinanderfolgenden Segmenten. Timestamps: start = erstes, end = letztes Segment im Chunk.


In [3]:
import numpy as np

def embed_segments(segments: list) -> list:
    """Embedded jedes Segment einzeln, gibt Liste von Vektoren zurück (gleiche Reihenfolge wie segments)."""
    texts = [s["text"] for s in segments]
    response = client.embeddings.create(model="text-embedding-3-small", input=texts)
    return [item.embedding for item in response.data]


def cosine_similarity(a: list, b: list) -> float:
    a, b = np.array(a), np.array(b)
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))


def chunk_by_semantic_similarity(segments: list, similarity_threshold: float = 0.5) -> list:
    """Gruppiert Segmente zu Chunks anhand thematischer Ähnlichkeit statt fester Zeitfenster.
    Bei Ähnlichkeits-Abfall unter den Threshold wird ein neuer Chunk begonnen."""
    embeddings = embed_segments(segments)

    chunks = []
    current_segments = [segments[0]]

    for i in range(1, len(segments)):
        sim = cosine_similarity(embeddings[i - 1], embeddings[i])

        if sim < similarity_threshold:
            # Themenwechsel erkannt -> aktuellen Chunk abschließen
            chunk_text = " ".join(s["text"] for s in current_segments)
            chunks.append({
                "start": current_segments[0]["start"],
                "end": current_segments[-1]["end"],
                "text": chunk_text,
            })
            current_segments = [segments[i]]
        else:
            current_segments.append(segments[i])

    # letzten Chunk nicht vergessen
    if current_segments:
        chunk_text = " ".join(s["text"] for s in current_segments)
        chunks.append({
            "start": current_segments[0]["start"],
            "end": current_segments[-1]["end"],
            "text": chunk_text,
        })

    return chunks

## Kalibrierung: similarity_threshold für semantisches Chunking messen

Berechnet die Similarity zwischen aufeinanderfolgenden Segmenten für einen kleinen Ausschnitt,
um an echten Werten (nicht geraten) zu sehen, wo Themenwechsel typischerweise liegen.


In [6]:
sample_segments = segments[:50]  # kleiner Ausschnitt, spart API-Calls
sample_embeddings = embed_segments(sample_segments)

for i in range(1, len(sample_segments)):
    sim = cosine_similarity(sample_embeddings[i - 1], sample_embeddings[i])
    print(f"{sim:.3f} | ...{sample_segments[i-1]['text'][-40:]} -> {sample_segments[i]['text'][:40]}...")

0.368 | ...- Welcome to the Huberman Lab Podcast, -> where we discuss science...
0.418 | ...where we discuss science -> and science-based tools for everyday lif...
0.199 | ...d science-based tools for everyday life. -> [upbeat rock music]...
0.168 | ...[upbeat rock music] -> I'm Andrew Huberman,...
0.330 | ...I'm Andrew Huberman, -> and I'm a professor of
neurobiology and ...
0.348 | ...fessor of
neurobiology and ophthalmology -> at Stanford School of Medicine....
0.167 | ...at Stanford School of Medicine. -> Today, we are talking all
about food and...
0.441 | ...re talking all
about food and the brain. -> We are going to talk about...
0.153 | ...We are going to talk about -> foods that are good for your
brain in te...
0.570 | ...e good for your
brain in terms of focus, -> in terms of brain health generally,...
0.593 | ...in terms of brain health generally, -> and the longevity of your brain,...
0.541 | ...and the longevity of your brain, -> your ability to maintain cognition...
0.440 

## Kalibrierung v2: Segmente zu Mini-Einheiten bündeln, dann vergleichen

Rohe Einzelsegmente sind zu kurz/fragmentarisch für zuverlässige Embedding-Vergleiche.
Bündelt je 5 Segmente zu einer Einheit, bevor Similarity gemessen wird.


In [7]:
def bundle_segments(segments: list, bundle_size: int = 5) -> list:
    bundles = []
    for i in range(0, len(segments), bundle_size):
        group = segments[i:i + bundle_size]
        bundles.append({
            "start": group[0]["start"],
            "end": group[-1]["end"],
            "text": " ".join(s["text"] for s in group),
        })
    return bundles

sample_bundles = bundle_segments(segments[:150], bundle_size=5)
bundle_embeddings = embed_segments(sample_bundles)

for i in range(1, len(sample_bundles)):
    sim = cosine_similarity(bundle_embeddings[i - 1], bundle_embeddings[i])
    print(f"{sim:.3f} | ...{sample_bundles[i-1]['text'][-50:]} -> {sample_bundles[i]['text'][:50]}...")

0.361 | ...day life. [upbeat rock music] I'm Andrew Huberman, -> and I'm a professor of
neurobiology and ophthalmol...
0.529 | ...ds that are good for your
brain in terms of focus, -> in terms of brain health generally, and the longev...
0.327 | ...ing over time. We are also going to talk about why -> and how you prefer
certain foods to others. And I'...
0.401 | ...hose are. One of those signals comes from your gut -> and is completely subconscious. This is not the gu...
0.367 | ...ding signals to your brain that you are unaware of -> about the nutrient contents of
the foods that you'...
0.573 | ...food can be converted into
energy that your brain, -> not your body, but that your brain can use. And th...
0.444 | ...al of belief. It's the signal of what you perceive -> and believe the food that
you're eating to contain...
0.568 | ...vague, but we're going to
provide mechanistic data -> to support the fact that
you can change what you e...
0.316 | ... you than the foods you might
current

## Finale Chunking-Funktion: adaptiver Perzentil-Threshold

Statt eines festen similarity_threshold wird die untere X-Perzentil-Grenze der gesamten
Similarity-Verteilung im Video berechnet -- passt sich automatisch an jedes Video an.


In [8]:
def chunk_by_semantic_similarity(segments: list, bundle_size: int = 5, percentile: float = 25) -> list:
    """Gruppiert Segmente zu Chunks anhand thematischer Ähnlichkeit.
    Schneidet dort, wo die Similarity im untersten X-Perzentil der Video-eigenen Verteilung liegt."""

    # Erst bündeln (Lehre aus der Kalibrierung: rohe Einzelsegmente sind zu fragmentarisch)
    bundles = bundle_segments(segments, bundle_size=bundle_size)
    embeddings = embed_segments(bundles)

    similarities = [
        cosine_similarity(embeddings[i - 1], embeddings[i])
        for i in range(1, len(embeddings))
    ]
    threshold = np.percentile(similarities, percentile)
    print(f"Adaptiver Threshold ({percentile}. Perzentil): {threshold:.3f}")

    chunks = []
    current = [bundles[0]]

    for i in range(1, len(bundles)):
        sim = similarities[i - 1]
        if sim < threshold:
            chunks.append({
                "start": current[0]["start"],
                "end": current[-1]["end"],
                "text": " ".join(b["text"] for b in current),
            })
            current = [bundles[i]]
        else:
            current.append(bundles[i])

    if current:
        chunks.append({
            "start": current[0]["start"],
            "end": current[-1]["end"],
            "text": " ".join(b["text"] for b in current),
        })

    return chunks

In [9]:
test_chunks = chunk_by_semantic_similarity(segments[:300])
print(f"{len(test_chunks)} Chunks aus 300 Segmenten (bzw. 60 Bundles)")
for c in test_chunks[:5]:
    print(f"[{c['start']:.1f}s-{c['end']:.1f}s]: {c['text'][:100]}...")

Adaptiver Threshold (25. Perzentil): 0.360
16 Chunks aus 300 Segmenten (bzw. 60 Bundles)
[0.4s-33.3s]: - Welcome to the Huberman Lab Podcast, where we discuss science and science-based tools for everyday...
[33.3s-108.9s]: and how you prefer
certain foods to others. And I'm going to talk about
the three major signals that...
[108.9s-241.3s]: This is an incredibly powerful
mechanism that we all have. It's one that I think is
very underapprec...
[241.3s-250.8s]: If it slides around a little
bit for social reasons or whatever reasons, it doesn't seem to be a big...
[250.8s-261.2s]: because of the way that
that feeding window impacts other genes called clock genes that regulate a b...


## Voller Lauf: semantisches Chunking auf allen 2500 Segmenten


In [10]:
semantic_chunks = chunk_by_semantic_similarity(segments)
print(f"\n{len(semantic_chunks)} semantische Chunks aus {len(segments)} Segmenten")

# Vergleich zur Baseline (zeitbasiert: 194 Chunks)
lengths = [c["end"] - c["start"] for c in semantic_chunks]
print(f"Chunk-Länge: min={min(lengths):.1f}s, max={max(lengths):.1f}s, avg={sum(lengths)/len(lengths):.1f}s")

Adaptiver Threshold (25. Perzentil): 0.409

126 semantische Chunks aus 2502 Segmenten
Chunk-Länge: min=5.4s, max=297.7s, avg=48.1s


## Neu-Indexierung in separate Test-Collection (Live-Daten bleiben unangetastet)

Speichert die semantischen Chunks in einer NEUEN Chroma-Collection, damit wir Alt vs. Neu
vergleichen können, ohne die Produktions-Daten zu riskieren.


In [11]:
def add_metadata_semantic(chunks: list, video_id: str, title: str) -> list:
    enriched = []
    for i, chunk in enumerate(chunks):
        enriched.append({
            **chunk,
            "chunk_id": f"{video_id}_semantic_{i}",
            "video_id": video_id,
            "title": title,
        })
    return enriched


semantic_chunks_with_metadata = add_metadata_semantic(
    semantic_chunks,
    video_id=video_id,
    title="Nutrients For Brain Health & Performance | Huberman Lab Podcast #42",
)

# Embeddings für die finalen Chunks (nicht die Bundle-Embeddings von vorhin -- neu, auf ganzen Chunk-Texten)
chunk_texts = [c["text"] for c in semantic_chunks_with_metadata]
response = client.embeddings.create(model="text-embedding-3-small", input=chunk_texts)
chunk_embeddings = [item.embedding for item in response.data]

import chromadb
chroma_client = chromadb.PersistentClient(path="../data/chroma_db")

# WICHTIG: neuer Collection-Name, NICHT "health_fitness_videos" (das ist die Live-Collection)
test_collection = chroma_client.get_or_create_collection(name="health_fitness_videos_semantic_test")

test_collection.upsert(
    ids=[c["chunk_id"] for c in semantic_chunks_with_metadata],
    embeddings=chunk_embeddings,
    documents=chunk_texts,
    metadatas=[{"video_id": c["video_id"], "title": c["title"], "start": c["start"], "end": c["end"]} for c in semantic_chunks_with_metadata],
)

print(f"✅ {test_collection.count()} semantische Chunks in Test-Collection gespeichert")

✅ 126 semantische Chunks in Test-Collection gespeichert


## Vergleichstest: dieselben 6 Fragen gegen die semantische Test-Collection

Direkter Vorher/Nachher-Vergleich mit identischem Setup wie bei der Baseline.


In [12]:
def search_test_collection(query: str, n_results: int = 2, max_distance: float = 1.0) -> list:
    emb = client.embeddings.create(model="text-embedding-3-small", input=[query]).data[0].embedding
    results = test_collection.query(
        query_embeddings=[emb],
        n_results=n_results,
        include=["documents", "metadatas", "distances"],
    )
    matches = []
    for doc, metadata, distance in zip(results["documents"][0], results["metadatas"][0], results["distances"][0]):
        if distance <= max_distance:
            matches.append({"text": doc, "start": metadata["start"], "end": metadata["end"]})
    return matches


semantic_results = {}
for q in baseline_questions:
    results = search_test_collection(q)
    semantic_results[q] = results
    print(f"--- {q} ---")
    for r in results:
        print(f"[{r['start']:.1f}s-{r['end']:.1f}s]: {r['text'][:150]}...")
    print()

--- What foods contain choline? ---
[1411.6s-1541.1s]: We have multiple clusters
of neurons in our brain that make acetylcholine. Two of the most prominent and well-known are the so-called nucleus basalis,...
[1610.3s-1740.0s]: It's really, yeah, incredible. They're using that as a source for all the building blocks of the body, but in particular, the nervous system. So eggs ...

--- How much creatine per day is recommended? ---
[2038.2s-2116.5s]: This was published just
very recently in 2021. And one thing to make clear, is that creatine supplementation has been shown to be especially useful fo...

--- What are anthocyanins found in? ---

--- Does fasting affect brain function? ---

--- What is the professor's academic title? ---

--- How does gut health relate to the brain? ---
[3864.9s-3932.0s]: of the actual taste on the mouth. Under normal conditions, it's a combination of the taste
of the thing on the mouth, plus the subconscious
signaling ...
[3944.8s-4061.5s]: They are certainl

## Judge-Bewertung: semantisches Chunking vs. Baseline


In [13]:
semantic_scores = {}
for q, chunks in semantic_results.items():
    verdict = judge_relevance(q, chunks)
    semantic_scores[q] = verdict
    print(f"--- {q} ---")
    print(verdict)
    print()

--- What foods contain choline? ---
RELEVANT: Die Textausschnitte nennen Eier, insbesondere Eigelb, als die Hauptquelle für Cholin und erwähnen auch pflanzliche Quellen wie Kartoffeln, Nüsse, Samen, Getreide und Obst, die ebenfalls Cholin enthalten.

--- How much creatine per day is recommended? ---
RELEVANT: Die Textausschnitte erwähnen, dass eine tägliche Supplementierung von mindestens fünf Gramm Kreatin empfohlen wird, um kognitive Vorteile zu erzielen.

--- What are anthocyanins found in? ---
NOT_RELEVANT: Kein Chunk gefunden

--- Does fasting affect brain function? ---
NOT_RELEVANT: Kein Chunk gefunden

--- What is the professor's academic title? ---
NOT_RELEVANT: Kein Chunk gefunden

--- How does gut health relate to the brain? ---
RELEVANT: Die Textausschnitte erläutern, wie eine gesunde Mikrobiota im Darm die neuronale Signalgebung an das Gehirn beeinflusst, was wiederum die Vorliebe für gesunde Nahrungsmittel fördert.



## Diagnose: Warum ist die Anthocyanin-Frage jetzt leer?

Schaut sich die RAW-Distances an (ohne Filter), um zu sehen ob es knapp am Threshold vorbeigeht
oder grundsätzlich kein guter Treffer mehr existiert.


In [14]:
emb = client.embeddings.create(model="text-embedding-3-small", input=["What are anthocyanins found in?"]).data[0].embedding
raw = test_collection.query(query_embeddings=[emb], n_results=5, include=["documents", "distances"])
for doc, dist in zip(raw["documents"][0], raw["distances"][0]):
    print(round(dist, 3), doc[:100])

1.009 you're going to see a
picture of a blueberry or some other berry, because of these anthocyanins. I p
1.04 that are beneficial for brain health is one that you've probably
seen pictures of online, because th
1.138 So that range of about 400 to
about 600 milligrams per day seems to be the minimum threshold for get
1.416 He's a absolutely
phenomenal neuroscientist at Columbia University in New York. Studies done by the 
1.434 and has been carried out in
parallel experiments in humans. This is largely, not exclusively, but la


## Option A: Threshold für die semantische Collection neu kalibrieren

Misst raw Distances für mehrere bekannte gute/schlechte Treffer in der neuen Collection,
um einen passenden max_distance-Wert zu finden (nicht raten, wie in Item 4).


In [15]:
calibration_queries = [
    "What foods contain choline?",       # bekannter guter Treffer erwartet
    "How much creatine per day is recommended?",  # bekannter guter Treffer erwartet
    "What are anthocyanins found in?",   # knapper Fall, Distance 1.009
    "What is the professor's academic title?",  # evtl. kein guter Treffer im Video
]

for q in calibration_queries:
    emb = client.embeddings.create(model="text-embedding-3-small", input=[q]).data[0].embedding
    raw = test_collection.query(query_embeddings=[emb], n_results=3, include=["documents", "distances"])
    print(f"--- {q} ---")
    for doc, dist in zip(raw["documents"][0], raw["distances"][0]):
        print(f"  {dist:.3f} | {doc[:80]}...")
    print()

--- What foods contain choline? ---
  0.885 | We have multiple clusters
of neurons in our brain that make acetylcholine. Two o...
  0.968 | It's really, yeah, incredible. They're using that as a source for all the buildi...
  1.085 | in supplementing with phosphatidylserine, it's a relatively
inexpensive suppleme...

--- How much creatine per day is recommended? ---
  0.840 | This was published just
very recently in 2021. And one thing to make clear, is t...
  1.026 | But nonetheless, I think it's interesting that creatine supplementation
of five ...
  1.029 | I generally consume these things like EPAs, creatine, alpha-GPC to set a general...

--- What are anthocyanins found in? ---
  1.009 | you're going to see a
picture of a blueberry or some other berry, because of the...
  1.040 | that are beneficial for brain health is one that you've probably
seen pictures o...
  1.138 | So that range of about 400 to
about 600 milligrams per day seems to be the minim...

--- What is the professor

In [16]:
def search_test_collection_v2(query: str, n_results: int = 2, max_distance: float = 1.1) -> list:
    emb = client.embeddings.create(model="text-embedding-3-small", input=[query]).data[0].embedding
    results = test_collection.query(query_embeddings=[emb], n_results=n_results, include=["documents", "metadatas", "distances"])
    matches = []
    for doc, metadata, distance in zip(results["documents"][0], results["metadatas"][0], results["distances"][0]):
        if distance <= max_distance:
            matches.append({"text": doc, "start": metadata["start"], "end": metadata["end"]})
    return matches

semantic_results_v2 = {q: search_test_collection_v2(q) for q in baseline_questions}
semantic_scores_v2 = {}
for q, chunks in semantic_results_v2.items():
    verdict = judge_relevance(q, chunks)
    semantic_scores_v2[q] = verdict
    print(f"--- {q} ---\n{verdict}\n")

--- What foods contain choline? ---
RELEVANT: Die Textausschnitte nennen Eier, insbesondere Eigelb, als die Hauptquelle für Cholin und erwähnen auch pflanzliche Quellen wie Kartoffeln, Nüsse, Samen, Getreide und Obst, die ebenfalls Cholin enthalten.

--- How much creatine per day is recommended? ---
RELEVANT: Die Textausschnitte empfehlen eine tägliche Einnahme von mindestens fünf Gramm Kreatin, insbesondere für Personen, die keine tierischen Produkte konsumieren.

--- What are anthocyanins found in? ---
RELEVANT: Die Textausschnitte beschreiben, dass Anthocyanine in Beeren wie Blaubeeren, Brombeeren und dunklen Johannisbeeren vorkommen und deren gesundheitliche Vorteile, insbesondere für die Gehirnfunktion.

--- Does fasting affect brain function? ---
NOT_RELEVANT: Die Textausschnitte konzentrieren sich auf allgemeine Gesundheitsfaktoren und die Rolle von Lebensmitteln für die Gehirnfunktion, behandeln jedoch nicht spezifisch die Auswirkungen des Fastens auf die Gehirnfunktion.

--- W

## Wiederverwendbare Funktion: Chunking-Methoden für neues Video vergleichen

Läuft beide Chunking-Methoden gegen ein neues Video, evaluiert mit gegebenen Testfragen,
gibt eine klare Empfehlung zurück. Übernimmt NICHT automatisch in die Live-Collection
(bewusste Entscheidung: letzter Schritt bleibt eine bewusste, manuelle Bestätigung).


In [17]:
def compare_chunking_methods(video_id: str, title: str, questions: list) -> dict:
    """Vergleicht zeitbasiertes vs. semantisches Chunking für ein Video.
    questions: 5-6 gezielte Testfragen zu Themen, die im Video sicher vorkommen (Platzhalter anpassen!).
    Gibt Empfehlung + beide fertigen Chunk-Sets zurück -- übernimmt NICHTS automatisch."""

    # 1. Transcript laden
    with open(f"../data/transcripts/{video_id}.json", "r", encoding="utf-8") as f:
        transcript_data = json.load(f)
    segments = transcript_data["segments"]
    cleaned = clean_segments(segments)

    # 2. Beide Chunking-Methoden
    time_chunks = add_metadata(chunk_by_time(cleaned), video_id, title)
    semantic_chunks = add_metadata_semantic(chunk_by_semantic_similarity(cleaned), video_id, title)

    # 3. Beide in temporäre Test-Collections indexieren
    chroma_client = chromadb.PersistentClient(path="../data/chroma_db")
    time_coll = chroma_client.get_or_create_collection(name=f"{video_id}_time_test")
    semantic_coll = chroma_client.get_or_create_collection(name=f"{video_id}_semantic_test")

    for coll, chunks in [(time_coll, time_chunks), (semantic_coll, semantic_chunks)]:
        texts = [c["text"] for c in chunks]
        embeds = [item.embedding for item in client.embeddings.create(model="text-embedding-3-small", input=texts).data]
        coll.upsert(
            ids=[c["chunk_id"] for c in chunks],
            embeddings=embeds,
            documents=texts,
            metadatas=[{"video_id": c["video_id"], "title": c["title"], "start": c["start"], "end": c["end"]} for c in chunks],
        )

    # 4. Beide mit denselben Fragen testen
    def search(coll, query, max_distance=1.1):
        emb = client.embeddings.create(model="text-embedding-3-small", input=[query]).data[0].embedding
        r = coll.query(query_embeddings=[emb], n_results=2, include=["documents", "distances"])
        return [doc for doc, dist in zip(r["documents"][0], r["distances"][0]) if dist <= max_distance]

    time_score, semantic_score = 0, 0
    print(f"=== Vergleich für Video {video_id} ===\n")
    for q in questions:
        time_chunks_found = search(time_coll, q)
        semantic_chunks_found = search(semantic_coll, q)
        time_verdict = judge_relevance(q, [{"text": t} for t in time_chunks_found])
        semantic_verdict = judge_relevance(q, [{"text": t} for t in semantic_chunks_found])
        time_pass = time_verdict.startswith("RELEVANT")
        semantic_pass = semantic_verdict.startswith("RELEVANT")
        time_score += time_pass
        semantic_score += semantic_pass
        print(f"[{q}]  Zeit: {'✅' if time_pass else '❌'}  Semantisch: {'✅' if semantic_pass else '❌'}")

    n = len(questions)
    print(f"\nZeitbasiert: {time_score}/{n}  |  Semantisch: {semantic_score}/{n}")

    winner = "semantic" if semantic_score > time_score else ("time" if time_score > semantic_score else "tie")
    print(f"Empfehlung: {winner.upper()}")

    return {
        "winner": winner,
        "time_score": time_score,
        "semantic_score": semantic_score,
        "time_chunks": time_chunks,
        "semantic_chunks": semantic_chunks,
    }

In [ ]:
result = compare_chunking_methods(
    video_id="NEUES_VIDEO_ID",
    title="Video-Titel hier",
    questions=[
        "Frage 1 zu einem Kern-Thema des Videos",
        "Frage 2 ...",
        "Frage 3 ...",
        "Frage 4 ...",
        "Frage 5 ...",
    ],
)